In [ ]:


import torch
import os, random
import numpy as np
from transformers import set_seed

SEED = 1561312


# 1) Python, NumPy, PyTorch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2) HF helper
set_seed(SEED)
generator = torch.Generator().manual_seed(SEED)

In [ ]:
model_path = "google/siglip-base-patch16-224"
device = "cuda"

In [ ]:
from transformers import SiglipProcessor

processor = SiglipProcessor.from_pretrained(model_path)

In [ ]:
from seq_aux_dataloader_aug import siglipFinetuner
from transformers import SiglipProcessor, SiglipModel, SiglipTokenizer, SiglipTextConfig
import torch

def load_base_model(processor, modelo_base):

    trained_model = SiglipModel.from_pretrained(
        model_path,
        device_map=device,
        # torch_dtype=torch.bfloat16
    )

    trained_model = torch.compile(trained_model)

    trained_model = siglipFinetuner(trained_model, processor, lr = 5e-5, only_projection=False)

    checkpoint = torch.load(f"/run/media/victor/pessoal/mestrado/codigo/train/checkpoint/V4_phase_1/{modelo_base}")
    trained_model.load_state_dict(checkpoint["state_dict"])
    trained_model = trained_model.siglip_model

    trained_model = trained_model.to(torch.bfloat16)

    return trained_model

In [ ]:
train_dataset = "/run/media/victor/pessoal/mestrado/codigo/datasets/v6_train_leiomyoma.csv"
val_dataset = "/run/media/victor/pessoal/mestrado/codigo/datasets/v6_val_leiomyoma.csv"

In [ ]:
torch.set_float32_matmul_precision('high')

In [ ]:
from seq_aux_dataloader_aug import imageTextDataset
from torch.utils.data import ConcatDataset

train_data = imageTextDataset(train_dataset, processor)

# train_data = imageTextDataset(train_dataset, processor)

val_data = imageTextDataset(val_dataset, processor)

In [ ]:
# len(configs)

In [ ]:
configs = [
 
    # {
    #     "lr" : 5e-5,
    #     "batch_size" : 100
    # },
    # {
    #     "lr" : 5e-5,
    #     "batch_size" : 200
    # },
    # {
    #     "lr" : 2.5e-5,
    #     "batch_size" : 150
    # },
    # {
    #     "lr" : 2.5e-5,
    #     "batch_size" : 200
    # },
    # {
    #     "lr" : 2.5e-5,
    #     "batch_size" : 100
    # },
    # {
    #     "lr" : 1e-5,
    #     "batch_size" : 100
    # },
    {
        "lr" : 1e-5,
        "batch_size" : 150
    },
    {
        "lr" : 7.5e-6,
        "batch_size" : 150
    },
    {
        "lr" : 5e-6,
        "batch_size" : 150
    },
    {
        "lr" : 1e-5,
        "batch_size" : 200
    },
    {
        "lr" : 7.5e-6,
        "batch_size" : 200
    },
    {
        "lr" : 5e-6,
        "batch_size" : 200
    },
 

]

In [ ]:
modelos_bases = [
    "siglip-epoch1-val_loss2.78238-batch100-lr5.00e-05-v1347-seed1561312-modelo_basephase_1.ckpt"
]

In [ ]:
from seq_aux_dataloader_aug import train
import gc

for modelo_base in modelos_bases:
    for config in configs:
        print(config)

        trained_model = load_base_model(processor, modelo_base)

        trained_model = train(train_data,val_data, trained_model, processor, lr=config["lr"], batch_size=config["batch_size"], num_workers = 2, seed=SEED, modelo_base=modelo_base, only_projection=False, folder="V6_phase_2_no_aug")

        del trained_model
        
        gc.collect()
        torch.cuda.empty_cache()
